# Notebook 00 — Setup & Orientation
## Sarvam AI × Indic NLP: A *Speech and Language Processing* Companion

**Purpose:** Verify your environment, meet the Sarvam model family, and see why Indic NLP is fundamentally different from the English-centric world of the textbook.

---
**Prerequisites:** Run `pip install -r ../requirements.txt` and create a `.env` file in the repo root:
```
SARVAM_API_KEY=<your_key_here>
DEMO_MODE=True
```

In [ ]:
# Cell 1 — Install/import check, load .env, init client
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from utils.sarvam_helpers import load_client, reset_demo_counters
from data.sample_texts import SAMPLE_TEXTS, LANGUAGE_NAMES, ENGLISH_TRANSLATIONS
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

reset_demo_counters()
client = load_client()
print('✓ Environment OK — SarvamAI client ready')
print(f'✓ DEMO_MODE = {os.getenv("DEMO_MODE")}')

## Why Indic NLP is Hard — Background & Framing

Jurafsky & Martin (*Speech and Language Processing* opens with the ambition: *"Every question you ever wanted to ask about language."* But the textbook's running examples are almost entirely **English**. When we shift to the Indian subcontinent, every assumption breaks:

| Challenge | English assumption | Indic reality |
|-----------|-------------------|---------------|
| **Scripts** | 1 script (Latin) | 10+ scripts (Devanagari, Tamil, Telugu, Bengali, Gujarati…) |
| **Morphology** | Relatively isolating | Agglutinative (Tamil, Telugu) or fusional (Hindi, Sanskrit) |
| **Word order** | SVO fixed | SOV default; Hindi allows scrambling |
| **Tokenization** | Whitespace works | Sandhi (Hindi), compound words (Tamil) break whitespace |
| **Code-mixing** | Rare | 10M+ speakers mix Hindi+English per sentence |
| **Resources** | 100B+ token corpora | Many languages: < 1B tokens |

**India has 22 scheduled languages** under the Constitution, written in at least 10 scripts. The top 8 languages by speaker count (Hindi, Bengali, Telugu, Tamil, Gujarati, Urdu, Kannada, Malayalam) together have **~1.2 billion speakers** — but most NLP research ignores them.

**Sarvam AI** was founded to fix this. Their model suite (Sarvam-M 24B, Bulbul TTS, Saaras STT, Mayura MT) is trained specifically on Indic language data from AI4Bharat corpora and proprietary sources.

In [ ]:
# Cell 3 — Language detection smoke test on all 4 sample texts
reset_demo_counters()
from utils.sarvam_helpers import detect_language

results = []
for lang_code, text in SAMPLE_TEXTS.items():
    try:
        det = detect_language(client, text)
        results.append({
            'Expected': LANGUAGE_NAMES[lang_code],
            'Expected Code': lang_code,
            'Detected': det['language_code'],
            'Confidence': f"{det['confidence']:.2%}",
            'Text Preview': text[:40] + '…'
        })
    except Exception as e:
        results.append({'Expected': lang_code, 'Error': str(e)})

df = pd.DataFrame(results)
print('Language Detection Results:')
print(df.to_string(index=False))

In [ ]:
# Cell 4 — Unicode codepoint range bar chart (script diversity visualization)
reset_demo_counters()

# Unicode block ranges for major Indic scripts
script_ranges = {
    'Latin (English)':   (0x0041, 0x007A),   # A-z
    'Devanagari (Hindi)': (0x0900, 0x097F),
    'Bengali':           (0x0980, 0x09FF),
    'Tamil':             (0x0B80, 0x0BFF),
    'Telugu':            (0x0C00, 0x0C7F),
    'Gujarati':          (0x0A80, 0x0AFF),
    'Kannada':           (0x0C80, 0x0CFF),
    'Malayalam':         (0x0D00, 0x0D7F),
    'Gurmukhi (Punjabi)':(0x0A00, 0x0A7F),
    'Odia':              (0x0B00, 0x0B7F),
}

labels = list(script_ranges.keys())
starts = [v[0] for v in script_ranges.values()]
sizes  = [v[1] - v[0] for v in script_ranges.values()]

colors = plt.cm.Set3.colors[:len(labels)]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(labels, sizes, left=starts, color=colors, edgecolor='white', height=0.7)
ax.set_xlabel('Unicode Codepoint')
ax.set_title('Unicode Block Ranges for Indic Scripts\n(width = codepoint count in block)', fontsize=13)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'U+{int(x):04X}'))
import seaborn as sns
sns.despine()
plt.tight_layout()
plt.savefig('../outputs/figures/00_unicode_ranges.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to outputs/figures/00_unicode_ranges.png')

## Sarvam Model Family Overview

| Model | Type | Key Capability | API Endpoint |
|-------|------|---------------|--------------|
| **Sarvam-M 24B** | LLM (reasoning) | 22 Indic languages, math/code, structured output | `chat.completions` |
| **Mayura v1** | Neural MT | High-quality translation, 4 style modes | `text.translate` |
| **Sarvam-Translate v1** | Neural MT | Alternative MT model for comparison | `text.translate` |
| **Bulbul v3** | TTS | 11 Indian languages, temperature-based expressiveness | `text_to_speech.convert` |
| **Bulbul v2** | TTS (legacy) | Pitch/loudness control (older API style) | `text_to_speech.convert` |
| **Saaras v3** | ASR | ~19% WER, 4 transcription modes | `speech_to_text.transcribe` |
| **Sarvam Vision 3B** | Multimodal | Document intelligence, structured extraction | `vision` |

### Model Selection Guide
- **Translation task** → Mayura v1 (formal/colloquial modes) or Sarvam-Translate v1
- **Text generation / reasoning in Hindi** → Sarvam-M 24B (`reasoning_effort=low` for speed)
- **Voice applications** → Bulbul v3 (temperature control) 
- **Transcription** → Saaras v3 (verbatim, transliterate, or code-mixed modes)
- **Document OCR + extraction** → Sarvam Vision 3B

### Textbook Connection
Each model maps to a chapter in the textbook:
- Mayura / Sarvam-Translate → **Ch 13** (Machine Translation)
- Sarvam-M → **Ch 10** (Transformers), **Ch 11** (Fine-tuning)
- Bulbul → **Ch 16** (TTS / Speech Synthesis)
- Saaras → **Ch 16** (ASR / Automatic Speech Recognition)

In [ ]:
# Cell 5 — Display sample texts with their English translations
reset_demo_counters()

print('='*70)
print('CANONICAL SAMPLE TEXTS — used throughout all notebooks')
print('='*70)
for lang_code, text in SAMPLE_TEXTS.items():
    lang_name = LANGUAGE_NAMES[lang_code]
    translation = ENGLISH_TRANSLATIONS[lang_code]
    print(f'\n[{lang_name} / {lang_code}]')
    print(f'  {text}')
    print(f'  → (EN) {translation}')
print('\n✓ Setup complete! Proceed to notebook 01_tokenization_morphology.ipynb')

---
## Krutrim AI — A Second Indic LLM Ecosystem

> **Note:** All Krutrim comparison cells require `KRUTRIM_CLOUD_API_KEY` set
> in `.env`. If you do not have a key, the cells will print a clear error and
> can be skipped — all Sarvam cells work independently.
>
> Get a key (with INR 10,000 free credits) at: cloud.olakrutrim.com

Krutrim is India's other major Indic AI platform, built by Ola (Bhavish Aggarwal).
Where Sarvam specialises in speech-first Indic APIs, Krutrim focuses on
**LLM reasoning** and **Indic embeddings** — making them complementary rather
than purely competing.

### Krutrim Model Family

| Model | Type | Key Capability | API |
|-------|------|---------------|-----|
| **Krutrim-spectre-v2** | LLM | Flagship chat/reasoning, 22 Indic languages | `chat.completions` |
| **Krutrim-2 (LLM-2)** | LLM (12B) | 128K context, Mistral-NeMo base, Indic fine-tuned | `chat.completions` |
| **Bhasantarit-mini** | Embeddings | 768-dim Indic semantic embeddings (Vyakyarth) | `embeddings` |
| **KrutrimTranslate** | Neural MT | 9 Indic languages <-> English | `language_labs.translate` |
| **Bhashik TTS** | Speech synthesis | Indic TTS via native SDK | `audio.speech.bhashik` |

### API Compatibility
Krutrim's `chat` and `embeddings` endpoints are **OpenAI-compatible** — the
same `openai` Python package works, just with a different `base_url`:
```python
from openai import OpenAI
client = OpenAI(
    api_key=os.environ["KRUTRIM_CLOUD_API_KEY"],
    base_url="https://cloud.olakrutrim.com/v1",
)
```
This means any code written for OpenAI's API runs on Krutrim with one line
changed — a significant ecosystem advantage for Indian developers.

### BharatBench: Krutrim-2 vs Competitors (published, 2024)

| Task | Krutrim-2 | Mistral-NeMo-12B | GPT-4 |
|------|-----------|-----------------|-------|
| IndicCOPA (commonsense) | **0.80** | 0.58 | ~0.78 |
| IndicSentiment | **0.95** | 0.71 | 0.90 |
| NER — Gujarati (error rate, lower is better) | **0.21** | 0.54 | 0.29 |
| NER — Kannada (error rate) | **0.20** | 0.31 | 0.25 |
| Hindi Retrieval (Vyakyarth) | **99.9%** | — | — |
| Bengali Retrieval | **98.7%** | — | — |

Krutrim-2 (12B parameters) matches or beats models 5-10x its size on most
Indic classification and retrieval tasks.


In [ ]:
# Notebook 00 — Krutrim API smoke test
import sys, os
sys.path.insert(0, os.path.abspath('..'))

try:
    from utils.krutrim_helpers import load_openai_client, krutrim_chat
    krutrim = load_openai_client()

    response = krutrim_chat(
        krutrim,
        messages=[{"role": "user",
                   "content": "Name the 4 Dravidian languages of India in one line."}],
    )
    print("Krutrim-spectre-v2 smoke test:")
    print(f"  {response.strip()}")
    print()

    # Side-by-side: same question to Sarvam-M
    from utils.sarvam_helpers import load_client, chat_complete, reset_demo_counters
    reset_demo_counters()
    sarvam = load_client()
    sarvam_resp = chat_complete(
        sarvam,
        [{"role": "user",
          "content": "Name the 4 Dravidian languages of India in one line."}],
    )
    if "<think>" in sarvam_resp:
        sarvam_resp = sarvam_resp.split("</think>")[-1].strip()

    print("Sarvam-M response:")
    print(f"  {sarvam_resp.strip()}")

except EnvironmentError as e:
    print(f"Krutrim key not configured: {e}")
    print("Set KRUTRIM_CLOUD_API_KEY in .env to enable Krutrim comparison cells.")
except Exception as e:
    print(f"Error: {e}")
